In [ ]:
import os

from pathlib import Path
os.chdir(Path.cwd().parent.parent)
import time
import copy
import json
import random
import warnings
import numpy as np
from pathlib import Path
from collections import defaultdict
from data.dataloader import load_dataset
from utils.llm import load_model
from modules.post_process import get_label_tokenizer
from modules.context import get_concepts
from utils.multiprocess import multiprocess_call
from modules.kg import get_neighbors

dataset = 'mimic3'
task = 'diagnosis'
task_full = {"drugrec": "drug recommendation", "diagnosis": "diagnosis prediction"}[task]
datasets, full_set = load_dataset(dataset, task)
test_set = datasets['test']
label_key = 'drugs' if task == 'drugrec' else 'label'
label_tokenizer = get_label_tokenizer(full_set, label_key)
all_candidates = list(label_tokenizer.vocabulary.token2idx.keys())
print("Total drugs:", len(all_candidates))

cache_dir = f"./TherapyReason/cache/{dataset}-{task}"
os.makedirs(cache_dir, exist_ok=True)

generator = "deepseek"
evaluator = load_model({"model": "Qwen3-235B-A22B", "server": "ant1", "do_sample": False, "temperature": 0.0, "max_attempts": 10})

# load data

In [ ]:
competitors = {
    "Distill": os.path.join(cache_dir, "interpret", "rationale", "deepseek"),  # 给定ground-truth生成的rationale
    "LLM": f"cache/{dataset}-{task}/CoT/2shot-deepseek-base-all",  # 使用LLM直接推理的reasoning
    "LRM": f"cache/{dataset}-{task}/CoT/2shot-deepseekr1-base-all",  # 使用LRM直接推理的reasoning
    "RAG:KARE": f"cache/{dataset}-{task}/KARE/2shot-deepseek-base-kg",  # 使用KARE生成的reasoning
    "RAG:MedRAG": f"cache/{dataset}-{task}/MedRAG/2shot-deepseek-base-kg",  # 使用MedRAG生成的reasoning
    "RAG:MedIKAL": f"cache/{dataset}-{task}/MedIKAL/0shot-deepseek-base-kg",  # 使用MedIKAL生成的reasoning
    "ours": [os.path.join(cache_dir, "sensitivity", "solution_thres4_dl", "deepseek"),
             os.path.join(cache_dir, "sensitivity", "solution_thres4_kg_2shot", "deepseek"),
             os.path.join(cache_dir, "sensitivity", "solution_thres4.jsonl")]
}


def gather_rationales(name):
    path = competitors[name]

    if task == 'diagnosis' and isinstance(path, str) and path.endswith("-all"):
        path = path.replace("-all", "-kg")

    rationale_samples = []
    if name == 'ours':
        solution_info = [json.loads(line) for line in Path(path[2]).read_text().splitlines()]

    for i, sample in enumerate(test_set):
        sample_id = f"{sample['patient_id']}_{sample['visit_id']}"
        if name == 'Distill':
            continue
            rationale = {}
            file = Path(os.path.join(path, f"{i}.json"))
            cache = json.loads(open(file).read())
            for entity, item in cache.items():
                if entity not in all_candidates: continue
                assert 'reason' in item and 'score' in item
                rationale[entity] = cache[entity]

        elif name in ['LLM', 'LRM', 'RAG:MedRAG', 'RAG:MedIKAL', 'RAG:KARE']:
            rationale = {}
            file = Path(os.path.join(path, f"{sample_id}.json"))
            cache = json.loads(open(file).read())
            for entity, item in cache['prediction'].items():
                if entity not in all_candidates: continue
                assert 'reason' in item and 'score' in item
                rationale[entity] = cache['prediction'][entity]

        elif name == 'ours':
            raw_rationale = defaultdict(dict)
            dl_cache = json.loads(Path(os.path.join(path[0], f"{i}.json")).read_text())
            kg_cache = json.loads(Path(os.path.join(path[1], f"{i}.json")).read_text())

            for key in ['dl', 'kg']:
                cache = {'dl': dl_cache, 'kg': kg_cache}
                for visit_id, visit in enumerate(cache[key]):
                    for solution_id, solution in enumerate(visit):
                        for entity, item in solution[f"{key}_prediction"].items():
                            if entity not in all_candidates: continue
                            assert 'reason' in item and 'predict' in item
                            header = solution_info[i]['solutions'][visit_id][solution_id]['header']
                            raw_rationale[entity][f"{visit_id}!!{solution_id}!!{header}"] = {"reason": item['reason'], 'score': int(item['predict'])}

            rationale = {}
            for entity, raw_item_dict in raw_rationale.items():
                agg_reason = ""
                for key, item in raw_item_dict.items():
                    visit_id, solution_id, header = key.split("!!")
                    agg_reason += f"Visit {visit_id}, Condition {solution_id} ({header}): {item['reason']}\n"
                agg_score = np.mean([item['score'] for item in raw_item_dict.values()]).round(3).item()
                rationale[entity] = {"reason": agg_reason, "score": agg_score}

        else:
            raise ValueError(f"Unknown competitor: {name}")

        rationale_samples.append(rationale)
    return rationale_samples

In [ ]:
rationale_competitors = {}
rationale_competitors['LLM'] = gather_rationales("LLM")
rationale_competitors['LRM'] = gather_rationales("LRM")
rationale_competitors['MedRAG'] = gather_rationales("RAG:MedRAG")
rationale_competitors['MedIKAL'] = gather_rationales("RAG:MedIKAL")
rationale_competitors['KARE'] = gather_rationales("RAG:KARE")
rationale_competitors['ours'] = gather_rationales("ours")

In [ ]:
def gather_supporting():
    supporting = []
    for i, sample in enumerate(test_set):
        conditions, procedures, drugs = get_concepts(sample, task, with_history=True, flatten=False)
        patient_context = ""
        for vid in range(len(conditions)):
            patient_context += f"## Visit {vid + 1}\n"
            if conditions[vid]:
                patient_context += "### Conditions\n" + "".join(x + "\n" for x in conditions[vid])
            if procedures[vid]:
                patient_context += "### Procedures\n" + "".join(x + "\n" for x in procedures[vid])
            if drugs[vid]:
                patient_context += "### Medications\n" + "".join(x + "\n" for x in drugs[vid])

        conditions_flat, procedures_flat, drugs_flat = get_concepts(sample, task, with_history=True, flatten=True)
        patient_concepts = conditions_flat + procedures_flat + drugs_flat

        supporting.append({"concepts": patient_concepts,
                           "context": patient_context,
                           "labels": sample[label_key]})

    return supporting


def get_relations(candidate, patient_concepts):
    relations = get_neighbors(candidate, patient_concepts, direction='both')
    relations = [f"{t[0].lower()}->{t[1].lower().replace('_', ' ')}->{t[2].lower()}" for t in relations]
    return relations

In [ ]:
supporting = gather_supporting()

In [ ]:
n_selected_samples = 200
n_selected_entities = 15


def prepare_compete_data(seed):
    random.seed(seed)
    random_sample_indices = random.sample(range(len(test_set)), n_selected_samples)

    items = []

    for i in random_sample_indices:
        context = supporting[i]['context']

        ground_pos_labels = [_ for _ in all_candidates if _ in supporting[i]['labels']]
        ground_neg_labels = [_ for _ in all_candidates if _ not in ground_pos_labels]

        #
        ground_pos_labels = [_ for _ in ground_pos_labels if _ in rationale_competitors['ours'][i]]
        ground_neg_labels = [_ for _ in ground_neg_labels if _ in rationale_competitors['ours'][i]]

        pos_entities = random.choices(ground_pos_labels, k=min(n_selected_entities, len(ground_pos_labels)))
        neg_entities = random.choices(ground_neg_labels, k=min(n_selected_entities, len(ground_neg_labels)))

        entity_items = []
        for entity in pos_entities + neg_entities:
            shuffle = random.choice([True, False])
            entity_items.append((entity, shuffle))

        items.append((i, context, entity_items))
    return items


compete_data = prepare_compete_data(seed=42)
print(f"Total {len(compete_data)} samples, {sum([len(item[2]) for item in compete_data])} entities")

# define compete functions

In [ ]:
def compete_factuality(context, candidate, relations, rationale_a, rationale_b, with_ref_rationale=False):
    # TODO with_ref_rationale: 是否使用distill的结果进行事实性评估
    prompt = f"""\
# Task
You are an expert in evaluating the factuality of {task_full} rationales. \
Given the patient's context, the candidate and relations between the candidate and patient's context, \
your task is to evaluate which rationale is more factual.

# Evaluation Criteria of Factuality
The rationale should be factual if it:
**Contains less sentences that are directly contradict to the given relations**

# Patient's Context
{context}

# Candidate and Relations
{candidate}
{relations}

# Rationales to Evaluate
## A
{rationale_a}
## B
{rationale_b}

# Evaluation Result (Provide brief reasoning and choose your result from the following options. **Nothing else**)
1. [A is more factual than B].
2. [B is more factual than A].
3. [Both are equally factual].
4. [Neither is factual].
"""

    def post_process_factuality(prob, response, output):
        if "A is more factual than B" in response:
            result = 1
        elif "B is more factual than A" in response:
            result = -1
        elif "Both are equally factual" in response or "Neither is factual" in response:
            result = 0
        else:
            raise ValueError(f"Invalid response: {response}")
        return prob, (response, result), output

    _, (response, result), _, usage = evaluator.understand(prompt, post_process=post_process_factuality)
    return result

In [ ]:
def compete_granularity(context, candidate, relations, rationale_a, rationale_b):
    prompt = f"""\
# Task
You are an expert in evaluating the granularity of {task_full} rationales. \
Given the patient's context and the candidate, your task is to evaluate which rationale is more granular.

# Evaluation Criteria of Granularity
The rationale should be granular if it:
1. Provides detailed reason about the candidate and its connection to the patient's context.
2. Includes specific data or evidence that supports the recommendations.

# Patient's Context
{context}

# Candidate
{candidate}

# Rationales to Evaluate
## A
{rationale_a}
## B
{rationale_b}

# Evaluation Result (Choose your result from the following options. **Nothing else**)
1. [A is more granular than B].
2. [B is more granular than A].
3. [Both are equally granular].
4. [Neither is granular].
"""

    def post_process_granularity(prob, response, output):
        if "A is more granular than B" in response:
            result = 1
        elif "B is more granular than A" in response:
            result = -1
        elif "Both are equally granular" in response or "Neither is granular" in response:
            result = 0
        else:
            raise ValueError(f"Invalid response: {response}")
        return prob, (response, result), output

    _, (response, result), _, usage = evaluator.understand(prompt, post_process=post_process_granularity)
    return result

In [ ]:
def compete_consistency(context, candidate, relations, rationale_a, rationale_b):
    prompt = f"""\
# Task
You are an expert in evaluating the consistency of {task_full} rationales. \
Given the patient's context and the candidate, your task is to evaluate which rationale is more consistent with the patient's context.

# Evaluation Criteria of Consistency
The rationale should be consistent if it:
1. The rationales are consistent with the patient's context.
2. The rationales are self-consistent with no internal contradiction.

# Patient's Context
{context}

# Candidate
{candidate}

# Rationales to Evaluate
## A
{rationale_a}
## B
{rationale_b}

# Evaluation Result (Choose your result from the following options. **Nothing else**)
1. [A is more consistent than B].
2. [B is more consistent than A].
3. [Both are equally consistent].
4. [Neither is consistent].
"""

    def post_process_consistency(prob, response, output):
        if "A is more consistent than B" in response:
            result = 1
        elif "B is more consistent than A" in response:
            result = -1
        elif "Both are equally consistent" in response or "Neither is consistent" in response:
            result = 0
        else:
            raise ValueError(f"Invalid response: {response}")
        return prob, (response, result), output

    _, (response, result), _, usage = evaluator.understand(prompt, post_process=post_process_consistency)
    return result

In [ ]:
def compete_fluency(context, candidate, relations, rationale_a, rationale_b):
    prompt = f"""\
# Task
You are an expert in evaluating the fluency of {task_full} rationales. \
Given the patient's context and the candidate, your task is to evaluate which rationale is more fluent.

# Evaluation Criteria of Fluency
The rationale should be fluent if it:
1. Is well-structured and easy to understand.
2. Avoids jargon or technical terms that may be confusing for non-experts.
3. Uses clear and concise language.

# Candidate
{candidate}

# Rationales to Evaluate
## A
{rationale_a}
## B
{rationale_b}

# Evaluation Result (Choose your result from the following options. **Nothing else**)
1. [A is more fluent than B].
2. [B is more fluent than A].
3. [Both are equally fluent].
4. [Neither is fluent].
"""

    def post_process_fluency(prob, response, output):
        if "A is more fluent than B" in response:
            result = 1
        elif "B is more fluent than A" in response:
            result = -1
        elif "Both are equally fluent" in response or "Neither is fluent" in response:
            result = 0
        else:
            raise ValueError(f"Invalid response: {response}")
        return prob, (response, result), output

    _, (response, result), _, usage = evaluator.understand(prompt, post_process=post_process_fluency)
    return result

In [ ]:
def compete_actionability(context, candidate, relations, rationale_a, rationale_b):
    assert task == 'drugrec'
    prompt = f"""\
# Task
You are an expert in evaluating the actionability of {task_full} rationales. \
Given the patient's context and the candidate, your task is to evaluate which rationale is more actionable.

# Evaluation Criteria of Actionability
The rationale should be actionable if it:
1. includes less maleficence drug-drug interaction and more beneficial drug-drug interaction.
2. includes less high-risk drugs and more low-risk drugs.

# Candidate
{candidate}

# Rationales to Evaluate
## A
{rationale_a}
## B
{rationale_b}

# Evaluation Result (Choose your result from the following options. **Nothing else**)
1. [A is more actionable than B].
2. [B is more actionable than A].
3. [Both are equally actionable].
4. [Neither is actionable].
"""

    def post_process_fluency(prob, response, output):
        if "A is more actionable than B" in response:
            result = 1
        elif "B is more actionable than A" in response:
            result = -1
        elif "Both are equally actionable" in response or "Neither is actionable" in response:
            result = 0
        else:
            raise ValueError(f"Invalid response: {response}")
        return prob, (response, result), output

    _, (response, result), _, usage = evaluator.understand(prompt, post_process=post_process_fluency)
    return result

In [ ]:
def compete_rationality(context, candidate, relations, rationale_a, rationale_b):
    assert task == 'diagnosis'
    prompt = f"""\
# Task
You are an expert in evaluating the rationality of {task_full} rationales. \
Given the patient's context and the candidate, your task is to evaluate which rationale is more based on clinical logic, medical knowledge, and standard diagnostic pathways.

# Evaluation Criteria of Rationality
The rationale should be rational if it:
1. Follows a logical "symptom -> sign -> pathophysiology -> disease hypothesis" chain
2. Prioritize the most discriminative symptoms, signs, and test results for the suspected condition
3. Is consistent with established clinical guidelines or classic diagnostic pathways

# Candidate
{candidate}

# Rationales to Evaluate
## A
{rationale_a}
## B
{rationale_b}

# Evaluation Result (Choose your result from the following options. **Nothing else**)
1. [A is more rational than B].
2. [B is more rational than A].
3. [Both are equally rational].
4. [Neither is rational].
"""

    def post_process_fluency(prob, response, output):
        if "A is more rational than B" in response:
            result = 1
        elif "B is more rational than A" in response:
            result = -1
        elif "Both are equally rational" in response or "Neither is rational" in response:
            result = 0
        else:
            raise ValueError(f"Invalid response: {response}")
        return prob, (response, result), output

    _, (response, result), _, usage = evaluator.understand(prompt, post_process=post_process_fluency)
    return result

In [ ]:
compete_functions = {
    "factuality": compete_factuality,
    "fluency": compete_fluency,
    "actionability": compete_actionability,
    "granularity": compete_granularity,
    "consistency": compete_consistency,
    "rationality": compete_rationality,
}

# compete

In [ ]:
def compete_single(key, item):
    i, context, entity_items, rationale_competitor_entities, rationale_guard_entities, fn, save_dir = item

    cache_file = Path(os.path.join(save_dir, f"{i}.json"))
    if cache_file.exists():
        return key, json.loads(cache_file.read_text())['scores']

    scores = []
    for j, (entity, shuffle) in enumerate(entity_items):
        time.sleep(random.uniform(0.5, 1))
        relations = get_relations(entity, supporting[i]['concepts'])
        relations = "\n".join(relations)

        if entity not in rationale_competitor_entities or entity not in rationale_guard_entities:
            continue
        rationale_a = rationale_competitor_entities[entity]['reason']
        rationale_b = rationale_guard_entities[entity]['reason']

        if shuffle:
            score = -fn(context, entity, relations, rationale_b, rationale_a)
        else:
            score = fn(context, entity, relations, rationale_a, rationale_b)
        scores.append(score)

    print(f"{i}: {np.unique(scores, return_counts=True)}")
    cache_file.write_text(json.dumps({"scores": scores, "entity_items": entity_items}))
    return key, scores

In [ ]:
compete_results = {}
if task == 'drugrec':
    metrics = ['factuality', 'granularity', 'consistency', 'actionability']
else:
    metrics = ['factuality', 'granularity', 'consistency', 'rationality']

for metric in metrics:
    print(metric)
    for competitor in ["LLM", 'LRM', 'MedRAG', 'MedIKAL', 'KARE']:
        print(competitor)
        guard = 'ours'
        rationale_samples_competitor = rationale_competitors[competitor]
        rationale_samples_guard = rationale_competitors[guard]
        save_dir = os.path.join(cache_dir, "interpret", f"{evaluator.genre}-{metric}", f"{competitor}-{guard}")
        os.makedirs(save_dir, exist_ok=True)

        items = []
        for i, context, entity_items in compete_data:
            items.append((i, context, entity_items, rationale_samples_competitor[i], rationale_samples_guard[i], compete_functions[metric], save_dir))

        compete_scores = multiprocess_call(compete_single, items, n_workers=4)
        compete_results[f"{competitor}-{guard}-{metric}"] = compete_scores

# print results

In [ ]:
for key, results in compete_results.items():
    # results: List[List[int]]
    flatten_results = [_ for result in results for _ in result]
    print(f"{key}: {np.unique(flatten_results, return_counts=True)}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcdefaults()

fontsize = 18
fig, axs = plt.subplots(1, 4, figsize=(20, 3), constrained_layout=True)

categories = ['Baseline wins', 'Draw', 'Ours wins']
metric2legend = {'factuality': 'Grounding', 'granularity': 'Granularity', 'consistency': 'Consistency', 'fluency': 'Fluency',
                 'actionability': "Safety", "rationality": "Rationality"}
competitors = ['LLM', 'LRM', 'MedRAG', 'MedIKAL', 'KARE']
competitor2legend = {"LLM": "CoT", "LRM": "Reason", "MedRAG": "MedRAG", "MedIKAL": "MedIKAL", "KARE": "KARE"}

category_colors = plt.colormaps['Set3']([3, 2, 6])

handles_labels = None
for mid, metric in enumerate(metrics):
    ax = axs[mid]
    data_dict = {}

    for competitor in competitors:
        guard = 'ours'
        results = compete_results[f"{competitor}-{guard}-{metric}"]
        flatten_results = np.array([_ for result in results for _ in result])

        _, counts = np.unique(flatten_results, return_counts=True)
        if len(counts) == 2:
            counts = np.array([counts[0], 0, counts[1]])
        data_dict[competitor] = counts[::-1]

    data = np.array([data_dict[comp] for comp in competitors])

    row_sums = data.sum(axis=1, keepdims=True)
    data_normalized = data / row_sums * 100
    data_cum_normalized = data_normalized.cumsum(axis=1)

    for i, (colname, color) in enumerate(zip(categories, category_colors)):
        widths = data_normalized[:, i]
        starts = data_cum_normalized[:, i] - widths
        rects = ax.barh(competitors, widths, left=starts, height=0.5,
                        label=colname, color=color)

        r, g, b, _ = color
        text_color = 'black'
        labels = [str(round(val, 1)) + "%" for val in data_normalized[:, i]]
        labels = [lab if lab != "0.0%" else "" for lab in labels]
        ax.bar_label(rects, label_type='center', color=text_color, labels=labels)

    ax.set_title([r'$\bf(a)$ ', r'$\bf(b)$ ', r'$\bf(c)$ ', r'$\bf(d)$ '][mid] + f'{metric2legend[metric].capitalize()}', fontsize=fontsize, y=-0.15)
    ax.invert_yaxis()
    ax.xaxis.set_visible(False)
    ax.set_xlim(0, 100)

    if mid != 0:
        ax.tick_params(axis='y', labelleft=False)

    if mid == 0:
        handles_labels = ax.get_legend_handles_labels()

for ax in axs:
    ax.set_yticks(np.arange(len(competitors)))
    ax.set_yticklabels([competitor2legend[_] for _ in competitors], fontsize=fontsize)

fig.legend(*handles_labels,
           loc='upper center',
           bbox_to_anchor=(0.5, 1.18),
           ncol=len(categories),
           fontsize=fontsize + 2,
           frameon=False)

# plt.show()
plt.savefig(f"Interpret_{evaluator.genre}_{task}.pdf", dpi=300, bbox_inches='tight', pad_inches=0)